# 03 — Calibrating the noise on one person's data

`tdee.calibrate.calibrate` fits phi, sigma_h, sigma_v and sigma_t by maximum likelihood (Nelder-Mead on the
innovations likelihood). Here: 30 synthetic users, 120 days each, simulated with parameters that differ from
the defaults; the fit starts from the defaults.

In [1]:
import sys
sys.path.insert(0, "../src")
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tdee import DEFAULT, Params, run, summarize, ols_tdee, SimConfig, simulate_user
from tdee.plotting import set_style, SERIES, INK_2
set_style()

In [2]:
from tdee.calibrate import calibrate

true = dict(phi=0.65, sigma_h=0.55, sigma_v=0.30)
rng = np.random.default_rng(3)
fits = []
for _ in range(30):
    inp, tr = simulate_user(SimConfig(days=120, spike_prob=0.0, **true), rng)
    p, _ = calibrate(inp, prior_t=tr["T"].iloc[0])
    fits.append({"phi": p.phi, "sigma_h": p.sigma_h, "sigma_v": p.sigma_v, "sigma_t": p.sigma_t})
fits = pd.DataFrame(fits)
summary = fits.describe().loc[["mean", "std", "25%", "50%", "75%"]].T
summary.insert(0, "true", [true["phi"], true["sigma_h"], true["sigma_v"], 5.0])
summary.round(3)

,true,mean,std,25%,50%,75%
phi,0.65,0.628,0.163,0.491,0.626,0.742
sigma_h,0.55,0.523,0.096,0.476,0.529,0.583
sigma_v,0.30,0.208,0.162,0.000,0.213,0.354
sigma_t,5.00,6.378,11.281,0.000,0.000,8.706


`sigma_t` (the TDEE drift) is weakly identified over a few months: the likelihood is flat in that direction
and the optimum often collapses toward 0. The library default keeps a small positive value (5 kcal/day) as a
modelling choice rather than a fitted one.